# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [11]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [12]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 125


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [13]:
# YOUR CHANGES HERE

def random_policy(state, actions):
    return 0

Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [14]:
# YOUR CHANGES HERE

for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
    print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)

CURR STATE 1089 ACTION 0 NEXT REWARD 0 NEXT STATE 1225
CURR STATE 1225 ACTION 0 NEXT REWARD 0 NEXT STATE 1361
CURR STATE 1361 ACTION 0 NEXT REWARD 0 NEXT STATE 1497
CURR STATE 1497 ACTION 0 NEXT REWARD 0 NEXT STATE 1633
CURR STATE 1633 ACTION 0 NEXT REWARD 0 NEXT STATE 1769
CURR STATE 1769 ACTION 0 NEXT REWARD 0 NEXT STATE 1905
CURR STATE 1905 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE 2041 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE 2041 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE 2041 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE 2041 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE 2041 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE 2041 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE 2041 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE 2041 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE 2041 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE 2041 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE 2041 ACTION 0 NEXT REWARD 0 NEXT STATE 2041
CURR STATE

In [15]:
import csv

with open("log-random.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")

    # Write the required column names
    writer.writerow([
        "curr_state",
        "curr_action",
        "next_reward",
        "next_state"
    ])

    # Run the simulator for 32 steps and save the results
    for curr_state, curr_action, next_reward, next_state in simulator.rollout_policy(
        random_policy, max_steps=32
    ):
        writer.writerow([
            curr_state,
            curr_action,
            next_reward,
            next_state
        ])

print("File saved: log-random.tsv")

File saved: log-random.tsv


Submit "log-random.tsv" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [ ]:
# YOUR CHANGES HERE

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=1024):
        #print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)
        pass

Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [16]:
# YOUR CHANGES HERE


import csv
import random

# Initialize the Q-table
q_table = {}

alpha = 1.0
gamma = 0.9

def get_q_value(state, action):
    return q_table.get((state, action), 0.0)

with open("q-random.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")

    writer.writerow([
        "curr_state",
        "curr_action",
        "next_reward",
        "next_state",
        "old_value",
        "new_value"
    ])

    for episode in range(32):
        for curr_state, curr_action, next_reward, next_state in simulator.rollout_policy(
            random_policy, max_steps=1024
        ):
            # Get the old Q-value
            old_value = get_q_value(curr_state, curr_action)

            # Find the maximum Q-value for the next state
            next_actions = simulator.get_actions(next_state)
            max_next_value = max(
                get_q_value(next_state, action)
                for action in next_actions
            )

            # Q-learning update: alpha = 1, gamma = 0.9
            new_value = old_value + alpha * (
                next_reward + gamma * max_next_value - old_value
            )

            # Store the updated Q-value
            q_table[(curr_state, curr_action)] = new_value

            # Save this transition and its Q-values
            writer.writerow([
                curr_state,
                curr_action,
                next_reward,
                next_state,
                old_value,
                new_value
            ])

print("Saved q-random.tsv")
print("Total transitions:", 32 * 1024)
print("Unique state-action pairs learned:", len(q_table))


Saved q-random.tsv
Total transitions: 32768
Unique state-action pairs learned: 215


Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [17]:
# YOUR CHANGES HERE

# hard-code epsilon=0.25. this is high but the environment is deterministic.
def epsilon_greedy_policy(state, actions):
    return 0

Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [18]:
# YOUR CHANGES HERE

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        #print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)

        if next_reward > 0:
            # moving to terminal state
            break

        pass

Submit "q-greedy.tsv" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [19]:
# YOUR CHANGES HERE


import random

epsilon = 0.25

def epsilon_greedy_policy(state, actions):
    # Explore: choose a random action 25% of the time
    if random.random() < epsilon:
        return random.choice(actions)

    # Exploit: choose the action with the highest Q-value
    q_values = [q_table.get((state, action), 0.0) for action in actions]
    max_q = max(q_values)

    # Randomly break ties between equally good actions
    best_actions = [
        action for action, value in zip(actions, q_values)
        if value == max_q
    ]
    return random.choice(best_actions)


In [20]:

import csv
import random

# Reset Q-learning state before training
q_table = {}

alpha = 1.0
gamma = 0.9
epsilon = 0.25

def epsilon_greedy_policy(state, actions):
    if random.random() < epsilon:
        return random.choice(actions)

    q_values = [q_table.get((state, action), 0.0) for action in actions]
    max_q = max(q_values)
    best_actions = [
        action for action, value in zip(actions, q_values)
        if value == max_q
    ]
    return random.choice(best_actions)

with open("q-greedy.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")

    writer.writerow([
        "curr_state",
        "curr_action",
        "next_reward",
        "next_state",
        "old_value",
        "new_value"
    ])

    for episode in range(32):
        for curr_state, curr_action, next_reward, next_state in simulator.rollout_policy(
            epsilon_greedy_policy, max_steps=1024
        ):
            old_value = q_table.get((curr_state, curr_action), 0.0)

            # The terminal state has no future reward
            if next_state == simulator.terminal_state:
                max_next_value = 0.0
            else:
                next_actions = simulator.get_actions(next_state)
                max_next_value = max(
                    q_table.get((next_state, action), 0.0)
                    for action in next_actions
                )

            new_value = old_value + alpha * (
                next_reward + gamma * max_next_value - old_value
            )

            q_table[(curr_state, curr_action)] = new_value

            writer.writerow([
                curr_state,
                curr_action,
                next_reward,
                next_state,
                old_value,
                new_value
            ])

            # Stop this episode when the rover reaches the terminal state
            if next_reward > 0:
                break

print("Saved q-greedy.tsv")
print("Unique state-action pairs learned:", len(q_table))


Saved q-greedy.tsv
Unique state-action pairs learned: 5021


In [21]:

import csv

with open("policy-greedy.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")

    # Required column names
    writer.writerow(["state", "action"])

    # Extract the best action for each non-terminal state
    for state in range(simulator.resolution * simulator.resolution * 8):
        if state == simulator.terminal_state:
            continue

        actions = simulator.get_actions(state)

        # Find the action with the highest learned Q-value
        q_values = [
            q_table.get((state, action), 0.0)
            for action in actions
        ]
        best_q = max(q_values)

        # Break ties randomly
        best_actions = [
            action
            for action, value in zip(actions, q_values)
            if value == best_q
        ]
        best_action = random.choice(best_actions)

        writer.writerow([state, best_action])

print("Saved policy-greedy.tsv")


Saved policy-greedy.tsv


Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [22]:
# YOUR CHANGES HERE


import csv
import random

# Reset Q-learning
q_table = {}
alpha = 1.0
gamma = 0.9
epsilon = 0.20

def training_policy(state, actions):
    # Explore 20% of the time
    if random.random() < epsilon:
        return random.choice(actions)

    # Otherwise, choose the best-known action
    values = [q_table.get((state, a), 0.0) for a in actions]
    best_value = max(values)
    best_actions = [
        a for a, value in zip(actions, values)
        if value == best_value
    ]
    return random.choice(best_actions)

# Train from every initial state repeatedly
training_rounds = 500
max_steps = 256
successful_episodes = 0

for episode in range(training_rounds):
    initial_states = simulator.initial_states.copy()
    random.shuffle(initial_states)

    for start_state in initial_states:
        curr_state = start_state

        for step in range(max_steps):
            actions = simulator.get_actions(curr_state)
            curr_action = training_policy(curr_state, actions)

            next_reward, next_state = simulator.get_next_reward_state(
                curr_state, curr_action
            )

            old_value = q_table.get((curr_state, curr_action), 0.0)

            # The terminal state has no future value
            if next_state == simulator.terminal_state:
                max_next_value = 0.0
            else:
                next_actions = simulator.get_actions(next_state)
                max_next_value = max(
                    q_table.get((next_state, a), 0.0)
                    for a in next_actions
                )

            # Update Q-value before checking for terminal reward
            new_value = old_value + alpha * (
                next_reward + gamma * max_next_value - old_value
            )
            q_table[(curr_state, curr_action)] = new_value

            if next_reward > 0:
                successful_episodes += 1
                break

            curr_state = next_state

# Extract a greedy policy for every state
with open("policy-optimal.tsv", "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["state", "action"])

    total_states = simulator.resolution * simulator.resolution * 8

    for state in range(total_states):
        if state == simulator.terminal_state:
            # Terminal state is absorbing; action does not matter
            action = 0
        else:
            actions = simulator.get_actions(state)
            values = [
                q_table.get((state, a), 0.0)
                for a in actions
            ]
            best_value = max(values)
            best_actions = [
                a for a, value in zip(actions, values)
                if value == best_value
            ]
            action = random.choice(best_actions)

        writer.writerow([state, action])

print("Saved policy-optimal.tsv")
print("Training episodes:", training_rounds * len(simulator.initial_states))
print("Successful episodes:", successful_episodes)
print("States in policy:", total_states)


Saved policy-optimal.tsv
Training episodes: 35500
Successful episodes: 35439
States in policy: 2048


Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.